In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (StrucToxNet)

This notebook processes and standardizes the **StrucToxNet** dataset by parsing peptide sequences derived from structural data, extracting toxicity labels encoded in sequence identifiers, resolving duplicate entries, and exporting a curated dataset for downstream machine learning applications.

- **Toxic effect / endpoint:** toxic
- **Source:** StrucToxNet
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:
- **Parses StrucToxNet FASTA files**, extracting peptide sequences and associated metadata.
- **Derives toxicity labels from sequence identifiers**:
  - labels are parsed directly from the FASTA header (`id` field), ensuring consistency with the original dataset definition.
- **Performs duplicate sequence checks**:
  - identical sequences with consistent labels are merged,
  - sequences with conflicting labels are flagged as erroneous.
- **Generates dataset-level metadata** using a centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `metadata.json`

In [2]:
name_source = "StrucToxNet"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*"):
    df = read_fasta_doc(file)
    df["label"] = df["id"].str.split("|").str[1].astype(int)
    df = df[["sequence", "label"]]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df.shape

(24570, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

- Working with metada

In [5]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [6]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 6, 5, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://github.com/jiaoshihu/StrucToxNet/tree/main',
 'publication': 'https://bmcbiol.biomedcentral.com/articles/10.1186/s12915-025-02329-1',
 'number_of_raw_sequences': 24570,
 'number_of_sequences_retained': 9544,
 'number_of_positive_sequences': 2491,
 'number_of_negative_sequences': 7053,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [7]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [8]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)